# 16 Differential Abundance and Diet

We use ANCOM-BC2 to test whether fungal genus abundance is associated with diet (`diet_type_sample`). We use the unrarefied genus count table from notebook 14 and keep the same samples for the unadjusted and adjusted models. The adjusted model accounts for age and settlement type, as in notebook 09.

## 1. Setup

### 1.1 Imports

In [5]:
import pandas as pd
import qiime2 as q2
from qiime2 import Artifact, Metadata

In [19]:
in_dir = "../data/processed_ignored/14_differential_abundance_setup"
out_dir = "../data/processed_ignored/16_differential_abundance_diet"
vis_dir = "../data/processed/16_differential_abundance_diet"
metadata = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"

unadjusted_dir = f"{vis_dir}/unadjusted_ancombc2" 
adjusted_dir = f"{vis_dir}/adjusted_ancombc2"

genus_table = f"{in_dir}/table_fungi_l6.qza"
diet_table = f"{out_dir}/table_fungi_l6_diet.qza"

diet_unadjusted_qza = f"{out_dir}/ancombc2_diet_unadjusted.qza"
diet_adjusted_qza = f"{out_dir}/ancombc2_diet_adjusted.qza"

diet_unadjusted_tabulate = f"{unadjusted_dir}/tabulate_unadjusted_diet.qzv"
diet_adjusted_tabulate = f"{adjusted_dir}/tabulate_adjusted_diet.qzv"

diet_unadjusted_barplot = f"{unadjusted_dir}/barplot_unadjusted_diet.qzv"
diet_adjusted_barplot = f"{adjusted_dir}/barplot_adjusted_diet.qzv"

!mkdir -p "$out_dir" "$vis_dir"
!mkdir -p "$unadjusted_dir" "$adjusted_dir"

### 1.3 Load inputs

We load the genus count table from notebook 14 and the cleaned sample metadata used in notebook 09.

In [7]:
genus_artifact = Artifact.load(genus_table)
sample_metadata = Metadata.load(metadata)

print("Table type:", genus_artifact.type)
print("Samples in metadata:", sample_metadata.id_count)

Table type: FeatureTable[Frequency]
Samples in metadata: 150


## 2. Select samples

### 2.1 Keep complete cases

We keep samples with settlement type, age, and diet information so both models use the same samples.

In [9]:
!qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$metadata" \
    --p-where "[Settlement_Type] IS NOT NULL AND [age_years_sample] IS NOT NULL AND [diet_type_sample] IS NOT NULL" \
    --o-filtered-table "$diet_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/16_differential_abundance_diet/table_fungi_l6_diet.qza


### 2.2 Summarize selected samples

We count the samples in each diet group after filtering for complete settlement, age, and diet information.

In [10]:
diet_counts = Artifact.load(diet_table).view(pd.DataFrame)
selected_metadata = sample_metadata.to_dataframe().loc[diet_counts.index]

print("Selected samples:", len(selected_metadata))
print(selected_metadata["diet_type_sample"].value_counts())

Selected samples: 139
diet_type_sample
Omnivore                            113
Vegetarian but eat seafood            8
Omnivore but do not eat red meat      7
Vegetarian                            6
Vegan                                 5
Name: count, dtype: int64


In [ ]:
!qiime composition ancombc2 --help

## 3. Run ANCOM-BC2

We use ANCOM-BC2 because some fungal genera were detected in only a few samples. When a genus has many zero counts, a result can depend on the small value used in place of zero. ANCOM-BC2 checks this by trying several small values and seeing whether the result stays significant. 

We add `--p-diff-robust` to both models to see which results stay significant when ANCOM-BC2 tries different small values for the zero counts.

### 3.1 Unadjusted ANCOM-BC2

We test whether fungal genus abundance differs by diet type. We use Omnivore as the reference group and compare Vegan, Vegetarian, "Vegetarian but eat seafood", and "Omnivore but dont eat red meat" samples with Omnivore samples.

In [14]:
!qiime composition ancombc2 \
    --i-table "$diet_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "diet_type_sample" \
    --p-reference-levels "diet_type_sample::Omnivore" \
    --p-diff-robust \
    --o-ancombc2-output "$diet_unadjusted_qza"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/16_differential_abundance_diet/ancombc2_diet_unadjusted.qza


In [15]:
!qiime composition tabulate \
    --i-data "$diet_unadjusted_qza" \
    --o-visualization "$diet_unadjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$diet_unadjusted_qza" \
    --o-visualization "$diet_unadjusted_barplot"


Saved Visualization to: ../data/processed/16_differential_abundance_diet/unadjusted_ancombc2/tabulate_unadjusted_diet.qzv
Saved Visualization to: ../data/processed/16_differential_abundance_diet/unadjusted_ancombc2/barplot_unadjusted_diet.qzv


In [17]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(diet_unadjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

for group in ["Vegetarian but eat seafood", "Omnivore but do not eat red meat", "Vegetarian", "Vegan"]:
    contrast = f"diet_type_sample::{group}"

    results = pd.DataFrame({
        "log_fold_change": tables["lfc"][contrast],
        "q_value": tables["q"][contrast],
        "diff_robust": tables["diff_robust"][contrast],
    })

    significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")
    print(f"{group} vs. Omnivore — {len(significant)} taxa with q < 0.05")
    display(significant)

Vegetarian but eat seafood vs. Omnivore — 2 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,-4.173109,0.003546,False
k__Fungi;p__Basidiomycota;c__Tremellomycetes;o__Filobasidiales;f__Filobasidiaceae;g__Naganishia,-3.712128,0.034312,False


Omnivore but do not eat red meat vs. Omnivore — 0 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,


Vegetarian vs. Omnivore — 1 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,3.839834,0.009804,False


Vegan vs. Omnivore — 0 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,


#### Unadjusted results

**Vegetarian but eat seafood vs. Omnivore**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| Dipodascaceae* | −4.173 | 0.003546 | False |
| *Naganishia* | −3.712 | 0.034312 | False |

*genus unidentified

**Vegetarian vs. Omnivore**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| Dipodascaceae* | +3.840 | 0.009804 | False |

*genus unidentified

**Comparisons where no taxa had q < 0.05**
- Omnivore but do not eat red meat vs. Omnivore
- Vegan vs. Omnivore

Compared with omnivores, the unidentified Dipodascaceae taxon had lower estimated abundance in vegetarians who ate seafood and higher estimated abundance in the Vegetarian group. *Naganishia* also had lower estimated abundance in vegetarians who ate seafood. None of these three results passed the sensitivity check for zero counts, so we interpret the differences cautiously.

### 3.2 Adjusted ANCOM-BC2

We compare Omnivore diet samples with other diet labels after accounting for age and settlement. We use the same 139 samples as in the unadjusted model.

In [20]:
!qiime composition ancombc2 \
    --i-table "$diet_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "age_years_sample + Settlement_Type + diet_type_sample" \
    --p-reference-levels "diet_type_sample::Omnivore" \
    --p-diff-robust \
    --o-ancombc2-output "$diet_adjusted_qza"

!qiime composition tabulate \
    --i-data "$diet_adjusted_qza" \
    --o-visualization "$diet_adjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$diet_adjusted_qza" \
    --o-visualization "$diet_adjusted_barplot"


Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/16_differential_abundance_diet/ancombc2_diet_adjusted.qza
Saved Visualization to: ../data/processed/16_differential_abundance_diet/adjusted_ancombc2/tabulate_adjusted_diet.qzv
Saved Visualization to: ../data/processed/16_differential_abundance_diet/adjusted_ancombc2/barplot_adjusted_diet.qzv


In [21]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(diet_adjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

for group in ["Vegetarian but eat seafood", "Omnivore but do not eat red meat", "Vegetarian", "Vegan"]:
    contrast = f"diet_type_sample::{group}"

    results = pd.DataFrame({
        "log_fold_change": tables["lfc"][contrast],
        "q_value": tables["q"][contrast],
        "diff_robust": tables["diff_robust"][contrast],
    })

    significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")
    print(f"{group} vs. Omnivore — {len(significant)} taxa with q < 0.05")
    display(significant)

Vegetarian but eat seafood vs. Omnivore — 2 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,-5.382307,0.000742,False
k__Fungi;p__Basidiomycota;c__Tremellomycetes;o__Trichosporonales;f__Trichosporonaceae;g__Apiotrichum,6.998400,0.003511,False


Omnivore but do not eat red meat vs. Omnivore — 1 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,-3.997502,0.010459,False


Vegetarian vs. Omnivore — 0 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,


Vegan vs. Omnivore — 0 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,


#### Adjusted results

**Vegetarian but eat seafood vs. Omnivore**

| Taxon | Log fold change | q-value | diff_robust |
|---|---:|---:|:---:|
| Dipodascaceae* | −5.382 | 0.000742 | False |
| *Apiotrichum* | +6.998 | 0.003511 | False |

*genus unidentified


**Omnivore but do not eat red meat vs. Omnivore**

| Taxon | Log fold change | q-value | diff_robust |
|---|---:|---:|:---:|
| Dipodascaceae* | −3.998 | 0.010459 | False |

*genus unidentified


**Comparisons where no taxa had q < 0.05**

- Vegetarian vs. Omnivore
- Vegan vs. Omnivore



After accounting for age and settlement type, the unidentified Dipodascaceae taxon had lower estimated abundance in both vegetarians who ate seafood and omnivores who avoided red meat compared with the Omnivore reference group. *Apiotrichum* had higher estimated abundance in vegetarians who ate seafood. None of these three results passed the sensitivity check for zero counts.


## 4. Discussion

Only one diet comparison showed the same taxon with q < 0.05 in both models. The unidentified Dipodascaceae taxon had lower estimated abundance in vegetarians who ate seafood than in omnivores, both before and after accounting for age and settlement type.

None of the findings passed the sensitivity check for zero counts. The Omnivore reference group also had 113 samples, while the other diet groups had only 5–8 each. Given these small groups and the sensitivity to zero counts, the evidence for diet-related differences in fungal abundance was limited.